In [2]:
%load_ext autoreload
%autoreload 2

import os, json, pickle
import torch
from torch.utils.data import DataLoader
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint

from go_ml.data_utils import *
from argparse import ArgumentParser
import transformers
from go_ml.models.func_cond_esm import FuncCondESM, FuncCondESMFinetune

from go_ml.data_utils import prot_func_collate_bert, ProtFuncDataset, BertFuncDataset
device = torch.device("cuda:2")

import pickle
data_path = "/home/andrew/GO_interp/data/train_esm_datasets/"
with open(f"{data_path}/train_dataset.pkl", "rb") as f:
    train_dataset = pickle.load(f)
with open(f"{data_path}/val_dataset.pkl", "rb") as f:
    val_dataset = pickle.load(f)

train_dataset = BertFuncDataset.from_prot_func_dataset(train_dataset)
val_dataset = BertFuncDataset.from_prot_func_dataset(val_dataset)

dataloader_params = {"shuffle": True, "batch_size": 6, "collate_fn": prot_func_collate_bert}
val_dataloader_params = {"shuffle": False, "batch_size": 12, "collate_fn": prot_func_collate_bert}

train_loader = DataLoader(train_dataset, **dataloader_params)
val_loader = DataLoader(val_dataset, **val_dataloader_params)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Get ESMC version running

In [ ]:
%load_ext autoreload
%autoreload 2

# from go_ml.models.func_cond_esmc import FuncCondESMC, FuncCondESMCFinetune
from transformers import AutoModelForMaskedLM, AutoTokenizer
model = AutoModelForMaskedLM.from_pretrained('Synthyra/ESMplusplus_small', torch_dtype='auto', trust_remote_code=True, map_location=device).train()

import os, json, pickle
import torch
from torch.utils.data import DataLoader
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint

from go_ml.data_utils import *
from argparse import ArgumentParser
import transformers
from go_ml.models.func_cond_esmc import FuncCondESMC, FuncCondESMCFinetune

from go_ml.data_utils import prot_func_collate_bert, ProtFuncDataset, BertFuncDataset

import pickle
data_path = "/home/andrew/GO_interp/data/train_esm_datasets/"
with open(f"{data_path}/train_dataset.pkl", "rb") as f:
    train_dataset = pickle.load(f)
with open(f"{data_path}/val_dataset.pkl", "rb") as f:
    val_dataset = pickle.load(f)

parser = ArgumentParser()
parser.add_argument(
            "--gpu_id",
            default=0,
            type=int,
            help="GPU ID to use for training",
        )
parser.add_argument(
            "--mask_func",
            default='perc',
            type=str,
            help="Mask func to use for input sequences: 'perc' for percentage masking, 'span' for span masking",
        )

parser = FuncCondESMCFinetune.add_model_specific_args(parser)
hparams = parser.parse_args([])

if(hparams.mask_func == 'perc'):
    mask_func = bert_mask_alias
elif(hparams.mask_func == 'span'):
    mask_func = bert_span_mask_alias
else:
    raise ValueError("Invalid mask_func. Choose 'perc' or 'span'.")

train_dataset = BertFuncDataset.from_prot_func_dataset(train_dataset, mask_func=mask_func)
val_dataset = BertFuncDataset.from_prot_func_dataset(val_dataset, mask_func=mask_func)

dataloader_params = {"shuffle": True, "batch_size": 10, "collate_fn": prot_func_collate_bert}
val_dataloader_params = {"shuffle": False, "batch_size": 12, "collate_fn": prot_func_collate_bert}

train_loader = DataLoader(train_dataset, **dataloader_params)
val_loader = DataLoader(val_dataset, **val_dataloader_params)

hparams.label_counts = train_dataset.labels.sum(axis=0).A1
hparams.num_train_steps = len(train_dataset)*15
hparams.learning_rate = 1e-6

model = FuncCondESMCFinetune(hparams)
batch = next(iter(train_loader))
(masked_seq_ind, masked_seq_labels, 
    mask, labels, seq_ind) = (batch['masked_seq_tensor'], batch['masked_seq_labels'], batch['seq_mask'], 
                        batch['labels'], batch['seq_tensor'])
labels = labels[:, model.active_labels] # only use active labels for func embedding
logits = model.forward(masked_seq_ind, mask, labels)